In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

path_2012 = "Tabelas5-sem_emprego_2012.csv"
path_2026 = "Tabelas5-sem_emprego_2026.csv"
path_horas = "Tabela 1.1.1.xls"

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
t2012 = pd.read_csv(path_2012, sep=";", decimal=",", encoding="utf-8-sig")
t2026 = pd.read_csv(path_2026, sep=";", decimal=",", encoding="utf-8-sig")

t2012 = t2012.rename(columns={
    "Desocupados - homens (2012 T1)": "homens_2012",
    "Desocupados - mulheres (2012 T1)": "mulheres_2012",
})
t2026 = t2026.rename(columns={
    "Desocupados - homens (2026 T1)": "homens_2026",
    "Desocupados - mulheres (2026 T1)": "mulheres_2026",
})

comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")

In [ ]:
indicador_1 = pd.read_excel(path_horas, sheet_name="2022", header=None, skiprows=8)
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.dropna(how="all").reset_index(drop=True)

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]
estados = estados.rename(columns={"uf_regiao": "Estado"}).reset_index(drop=True)

In [ ]:
comp = comp.merge(estados, on="Estado", how="inner")
print("Linhas em comp após o cruzamento:", comp.shape[0])
comp.head()

In [ ]:
ordem = comp.sort_values("mulheres_2026", ascending=False)["Estado"]

longo = comp.melt(
    id_vars=["Sigla", "Código", "Estado"],
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao_mulheres",
)
longo["Ano"] = longo["Ano"].map({"mulheres_2012": "2012 T1", "mulheres_2026": "2026 T1"})

fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(
    data=longo,
    y="Estado",
    x="Participacao_mulheres",
    hue="Ano",
    order=ordem,
    ax=ax,
)
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Participação das mulheres entre as pessoas desocupadas (%)")
ax.set_ylabel("")
ax.set_title("Desocupação: participação feminina em 2012 T1 e 2026 T1")
ax.legend(title="Trimestre")
fig.tight_layout()
fig.savefig("participacao_mulheres_desocupacao.png", dpi=150)
plt.show()

In [ ]:
comp.to_csv("comp_tabela5_x_tabela1_1_1.csv", index=False)